# 06 Summary Report

Generates a compact, reproducible report across repository audits, paired optimizer comparisons, generalization metrics, spectral diagnostics, and scaling readiness.

In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'wwgpt').exists():
            return p
    raise RuntimeError('Could not locate repository root')

REPO_ROOT = find_repo_root(Path.cwd())
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wwgpt.analysis import (
    completed_runs, discover_experiment_runs, normalize_metrics, terminal_results,
    add_generalization_measures, vocab_size_from_artifacts, summary, align_curves,
    paired_curve_differences,
)

RESULTS_ROOT = Path(globals().get('RESULTS_ROOT', REPO_ROOT / 'results'))
ANALYSIS_DIR = Path(globals().get('ANALYSIS_DIR', RESULTS_ROOT / 'notebook_analysis'))
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository: {REPO_ROOT}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Notebook outputs: {ANALYSIS_DIR}')


## Collect analysis tables

In [ ]:
runs=discover_experiment_runs(RESULTS_ROOT)
terminal=terminal_results(runs)
terminal.to_csv(ANALYSIS_DIR/'summary_paired_terminal_results.csv', index=False)
metrics=[]
for r in runs:
    m=normalize_metrics(r['artifacts'].get('metrics.csv', pd.DataFrame()))
    if not m.empty:
        m=add_generalization_measures(m, vocab_size_from_artifacts(r['artifacts']))
        m['optimizer']=r['optimizer']; m['seed']=r['seed']; m['pair_id']=r['pair_id']; metrics.append(m)
metrics_df=pd.concat(metrics, ignore_index=True) if metrics else pd.DataFrame()
metrics_df.to_csv(ANALYSIS_DIR/'summary_metrics_with_generalization.csv', index=False)
print(f'runs discovered: {len(runs)}; metric rows: {len(metrics_df)}; paired terminal rows: {len(terminal)}')
display(terminal)


## Primary paired comparison

In [ ]:
report_lines=[]
if terminal.empty or 'wwpgd_minus_adamw_final_validation_loss' not in terminal:
    report_lines.append('No complete paired terminal comparison is available yet.')
else:
    complete=terminal.dropna(subset=['wwpgd_minus_adamw_final_validation_loss'])
    diff=complete['wwpgd_minus_adamw_final_validation_loss']
    if diff.empty:
        report_lines.append('Paired rows exist, but no complete AdamW/WW-PGD terminal differences are available.')
    else:
        report_lines.append(f'Complete pairs: {len(diff)}.')
        report_lines.append(f'Mean WW-PGD minus AdamW final validation loss: {diff.mean():.6g} (negative favors WW-PGD).')
        report_lines.append(f'Median paired difference: {diff.median():.6g}.')
        report_lines.append(f'WW-PGD lower in {(diff < 0).sum()} pairs; AdamW lower in {(diff > 0).sum()} pairs; ties in {(diff == 0).sum()} pairs.')
        paired_summary=pd.DataFrame([summary(diff)]); paired_summary.to_csv(ANALYSIS_DIR/'summary_primary_paired_difference.csv', index=False); display(paired_summary)
display(Markdown('\n\n'.join(f'- {line}' for line in report_lines)))


## Generalization and overfitting synopsis

In [ ]:
if metrics_df.empty:
    display(Markdown('No metrics are available.'))
else:
    final=metrics_df.sort_values('step').groupby(['pair_id','optimizer'], dropna=False).tail(1) if 'step' in metrics_df else metrics_df.groupby(['pair_id','optimizer'], dropna=False).tail(1)
    cols=[c for c in ['val_loss','train_loss','generalization_gap','val_perplexity','val_token_prediction_capacity','capacity_generalization_gap'] if c in final]
    gen_summary=final.groupby('optimizer')[cols].agg(['count','mean','std','min','max']) if cols else pd.DataFrame()
    gen_summary.to_csv(ANALYSIS_DIR/'summary_generalization.csv')
    display(gen_summary)


## Executive report file

In [ ]:
md_lines=['# WW-PGD nanoGPT Notebook Summary', '', f'Results root: `{RESULTS_ROOT}`', f'Runs discovered: {len(runs)}', '']
md_lines += report_lines if 'report_lines' in globals() else ['Primary paired comparison not computed.']
if not metrics_df.empty:
    md_lines += ['', f'Metric records analyzed: {len(metrics_df)}']
if terminal.empty:
    md_lines += ['', 'No complete paired terminal result table is available yet.']
else:
    md_lines += ['', 'Paired terminal result preview:', '', terminal.head().to_markdown(index=False)]
report_path=ANALYSIS_DIR/'summary_report.md'
report_path.write_text('\n'.join(md_lines))
print(f'Wrote {report_path}')
display(Markdown(report_path.read_text()))
